<a href="https://colab.research.google.com/github/ulysis022219/AudioToText-WhisperX/blob/main/AudioToText_WhisperX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ AudioToText — WhisperX

**Turn any audio or video into text, subtitles, or a translation — right in your browser.**

Powered by [WhisperX](https://github.com/m-bain/whisperX) (faster Whisper with word-level timestamps). Nothing to install on your computer — it runs free on Google Colab's GPU.

---

### ▶️ How to use

1. **Turn on the GPU:** menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Run each step **in order, top to bottom** — click the round **▶️** button on the left of every cell.
3. Wait for a ✅ before moving to the next step.

> First time here? Just follow Steps 1 → 2 → 3 → 4. Step 5 is an optional extra.


## 1️⃣ Install (run once)

Click **▶️** below to install the pinned Colab environment. This takes a few minutes the first time.

> ⚠️ **The runtime will restart by itself after package verification. This is expected.** Continue to **Step 2** once it reconnects.


In [ ]:
#@title { display-mode: "form" }
# Pinned for Colab CPython 3.13. Verification uses package metadata because
# importing a replaced Torch stack in the old kernel can crash before restart.
import os
import shutil
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

FLAG = "/tmp/deps_installed_v5"
EXPECTED = {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "torchcodec": "0.7.0+cu128",
    "whisperx": "3.8.6",
    "openai": "3.6.0",
    "deepl": "1.32.0",
    "numpy": "2.5.2",
    "ctranslate2": "4.8.2",
}

def sh(desc, args):
    print(f"\n⏳ {desc}...")
    started = time.time()
    result = subprocess.run(args, capture_output=True, text=True)
    if result.returncode:
        print(f"❌ FAILED: {desc}")
        print(result.stdout[-1500:])
        print("--- STDERR ---")
        print(result.stderr[-3000:])
        raise RuntimeError(f"Step failed: {desc}")
    print(f"✅ {desc} ({time.time() - started:.0f}s)")
    return result

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None

def verify_environment():
    problems = []
    print("\n🔍 Verifying installed packages...")
    for package, expected in EXPECTED.items():
        actual = installed_version(package)
        if actual == expected:
            print(f"  ✅ {package} {actual}")
        else:
            problems.append(f"{package}: expected {expected}, found {actual or 'missing'}")
    if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode:
        problems.append("ffmpeg executable: missing")
    else:
        print("  ✅ ffmpeg executable")
    if problems:
        raise RuntimeError("Environment verification failed:\n  - " + "\n  - ".join(problems))

# whisperx 3.8.6 needs Python < 3.14 and numpy 2.5 needs >= 3.12; fail clearly, not deep in the resolver.
if not (3, 12) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Python {sys.version.split()[0]} is not supported. Use Colab's latest runtime "
                       "(Runtime → Change runtime type → Runtime version: Latest).")

if not os.path.exists(FLAG):
    if subprocess.run(["ffmpeg", "-version"], capture_output=True).returncode:
        if sys.platform == "linux":
            sh("Installing ffmpeg", ["apt-get", "install", "-y", "ffmpeg"])
        else:
            raise RuntimeError("Install ffmpeg first: https://ffmpeg.org/download.html")
    else:
        print("✅ ffmpeg already present")

    # Colab ships uv; install it only when missing.
    if shutil.which("uv"):
        uv = ["uv"]
    else:
        sh("Installing uv", [sys.executable, "-m", "pip", "install", "-q", "uv==0.8.22"])
        uv = [sys.executable, "-m", "uv"]
    # One resolve for everything: uv downloads in parallel and skips packages
    # that already match, e.g. a Colab image that already has this Torch.
    sh("Installing PyTorch cu128, WhisperX and backends", uv + [
        "pip", "install", "--system", "--python", sys.executable,
        *(f"{name}=={pinned}" for name, pinned in EXPECTED.items()),
        "--extra-index-url", "https://download.pytorch.org/whl/cu128",
        "--index-strategy", "unsafe-best-match",
    ])
    verify_environment()
    with open(FLAG, "w", encoding="utf-8") as handle:
        handle.write("verified")
    print("\n✅ Installation complete. Restarting runtime...")
    sys.stdout.flush()
    os.kill(os.getpid(), 9)
else:
    verify_environment()
    print("\n✅ Dependencies verified. Continue to Step 2.")


## 2️⃣ Add your audio files

Click **▶️** below to connect Google Drive. The notebook creates and lists:

```
📁 My Drive › for process
```

Copy a printed file path (or the folder path, to process everything) into `audio_file` in Step 3. Files and later outputs remain in Drive between Colab sessions.


In [ ]:
#@title 📁 Connect Google Drive { display-mode: "form" }
#@markdown Run this cell, approve Drive access, then add audio/video files.
#@markdown Files remain in Drive between sessions.
drive_folder = "/content/drive/MyDrive/for process" #@param {type:"string"}

from google.colab import drive
import os

drive.mount("/content/drive")
os.makedirs(drive_folder, exist_ok=True)
print(f"\n✅ Drive connected. Put files here: {drive_folder}\n")
files = sorted(
    os.path.join(drive_folder, name)
    for name in os.listdir(drive_folder)
    if os.path.isfile(os.path.join(drive_folder, name))
)
if files:
    print("Files found (copy a path into Step 3):")
    for filename in files:
        print("  •", filename)
else:
    print("Folder is empty — add files in Drive or the Colab Files panel.")


## 3️⃣ 🎬 Transcribe or translate

1. Choose the **local WhisperX** backend (audio stays in Colab) or explicitly choose **OpenAI API** (audio is uploaded to OpenAI).
2. Paste a file path from Step 2 — or the folder path to transcribe every audio/video file in it.
3. Click **▶️**. Progress is shown while each file runs; segments print when transcription completes.

Raw results are checkpointed to Google Drive after every completed file, so Step 4 can recover after a runtime interruption. The pinned WhisperX VAD checkpoint is hash-verified before its narrowly scoped legacy load; forced weights-only loading remains enabled for alignment models.


In [ ]:
#@title 🎬 Step 3 — Transcribe / Translate { display-mode: "form" }
import contextlib
import gc
import hashlib
import io
import json
import logging
import math
import os
os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"
import re
import shutil
import subprocess
import tempfile
import warnings
import zlib
from pathlib import Path

#@markdown ### Backend and privacy
#@markdown Local WhisperX keeps audio in Colab. OpenAI API uploads audio to OpenAI.
transcription_backend = "Local WhisperX (Colab GPU)" #@param ["Local WhisperX (Colab GPU)", "OpenAI API (uploads audio)"]
#@markdown ### Task
task_label = "Transcribe" #@param ["Transcribe", "Translate to English"]
task = "transcribe" if task_label == "Transcribe" else "translate"
#@markdown ### Input — a file path, or a folder to process every audio/video file in it
#@markdown Commas are allowed in filenames. Several paths can be given one per line in code.
audio_file = "" #@param {type:"string"}
#@markdown ### Language and quality
language = "Japanese" #@param ["Auto-Detect", "Afrikaans", "Albanian", "Amharic", "Arabic", "Armenian", "Assamese", "Azerbaijani", "Bashkir", "Basque", "Belarusian", "Bengali", "Bosnian", "Breton", "Bulgarian", "Catalan", "Chinese", "Croatian", "Czech", "Danish", "Dutch", "English", "Estonian", "Faroese", "Finnish", "French", "Galician", "Georgian", "German", "Greek", "Gujarati", "Haitian Creole", "Hausa", "Hawaiian", "Hebrew", "Hindi", "Hungarian", "Icelandic", "Indonesian", "Italian", "Japanese", "Javanese", "Kannada", "Kazakh", "Khmer", "Korean", "Lao", "Latin", "Latvian", "Lingala", "Lithuanian", "Luxembourgish", "Macedonian", "Malagasy", "Malay", "Malayalam", "Maltese", "Maori", "Marathi", "Mongolian", "Myanmar", "Nepali", "Norwegian", "Nynorsk", "Occitan", "Pashto", "Persian", "Polish", "Portuguese", "Punjabi", "Romanian", "Russian", "Sanskrit", "Serbian", "Shona", "Sindhi", "Sinhala", "Slovak", "Slovenian", "Somali", "Spanish", "Sundanese", "Swahili", "Swedish", "Tagalog", "Tajik", "Tamil", "Tatar", "Telugu", "Thai", "Tibetan", "Turkish", "Turkmen", "Ukrainian", "Urdu", "Uzbek", "Vietnamese", "Welsh", "Yiddish", "Yoruba"]
#@markdown `kotoba-whisper-v2.0` is a Japanese-only distillation of large-v3: ~6x faster, transcribe only.
#@markdown `anime-whisper` is Kotoba fine-tuned on voice-acted Japanese: whispers, breaths, emotive speech and
#@markdown punctuation. Transcribe only; converted once (~5 min) and cached on Drive.
use_model = "large-v3" #@param ["large-v2", "large-v3", "kotoba-whisper-v2.0", "anime-whisper"]
quality_mode = "Balanced" #@param ["Balanced", "High accuracy"]
#@markdown Quiet speech: more sensitive voice detection and shorter chunks, for whispering/ASMR.
quiet_speech = False #@param {type:"boolean"}
#@markdown Block exact loops (a word hallucinated dozens of times) and shorten any run of 8+ repeats
#@markdown to four (るるるる…). Can also trim real repetition.
reduce_repetition = False #@param {type:"boolean"}
#@markdown ### Terminology hints (optional)
#@markdown Names, terms or acronyms Whisper keeps mis-hearing, e.g. "田中, ChatGPT".
#@markdown Keep it short: Whisper can echo hint text over quiet audio.
prompt = "" #@param {type:"string"}
#@markdown ### Advanced
compute_type = "float16" #@param ["float16", "int8"]
batch_size = 8 #@param {type:"integer"}
word_timestamps = True #@param {type:"boolean"}
fail_if_alignment_fails = False #@param {type:"boolean"}
max_chars_per_line = 20 #@param {type:"integer"}

MEDIA_EXTENSIONS = {
    ".aac", ".aif", ".aiff", ".amr", ".flac", ".m4a", ".mka", ".mkv", ".mov", ".mp3",
    ".mp4", ".mpeg", ".mpga", ".oga", ".ogg", ".opus", ".wav", ".webm", ".wma", ".wmv",
}

def resolve_audio_inputs(value):
    """Expand one path/folder per line into a de-duplicated list of media files."""
    resolved = []
    for line in value.splitlines():
        entry = line.strip().strip("\"'").strip()  # tolerate pasted quotes
        if not entry:
            continue
        if os.path.isdir(entry):
            found = sorted(
                str(path) for path in Path(entry).iterdir()
                if path.is_file() and not path.name.startswith(".")
                and path.suffix.lower() in MEDIA_EXTENSIONS)
            if not found:
                raise FileNotFoundError(f"No audio/video files found in folder: {entry}")
            resolved.extend(found)
        elif os.path.isfile(entry):
            resolved.append(entry)
        else:
            raise FileNotFoundError(f"File not found: {entry}")
    if not resolved:
        raise ValueError("audio_file is empty — paste a file or folder path from Step 2")
    return list(dict.fromkeys(resolved))

audio_files = resolve_audio_inputs(audio_file)
print(f"{len(audio_files)} input file(s) queued.")
if batch_size < 1 or max_chars_per_line < 1:
    raise ValueError("batch_size and max_chars_per_line must be positive")

effective_prompt = prompt.strip()

quality_presets = {
    "Balanced": {"beam_size": 3},
    "High accuracy": {"beam_size": 5},
}
if quality_mode not in quality_presets:
    raise ValueError(f"Unknown quality mode: {quality_mode}")
quality = quality_presets[quality_mode]

from google.colab import userdata
from whisperx.utils import TO_LANGUAGE_CODE as WHISPER_LANGUAGES
kotoba_based = use_model in {"kotoba-whisper-v2.0", "anime-whisper"}
if kotoba_based:
    if task != "transcribe":
        raise ValueError(f"{use_model} cannot translate. Use large-v3, or Transcribe then Step 5.")
    language = "Japanese"  # it cannot detect or emit other languages
if use_model == "kotoba-whisper-v2.0":
    # ponytail: follows the model's main branch, like the stock Whisper sizes do (CTranslate2 weights, no pickle).
    use_model = "kotoba-tech/kotoba-whisper-v2.0-faster"
elif use_model == "anime-whisper" and effective_prompt:
    print("ℹ️ anime-whisper ignores prompt: its authors report hints make it hallucinate.")
    effective_prompt = ""
lang_code = None if language == "Auto-Detect" else WHISPER_LANGUAGES.get(language.lower())
if language != "Auto-Detect" and not lang_code:
    raise ValueError(f"Unsupported language: {language}")

CHECKPOINT_PATH = "/content/drive/MyDrive/audio_transcription/_last_results.json"
CJK_LANGUAGE_CODES = {"ja", "zh"}
WHISPERX_VAD_SHA256 = "0b5b3216d60a2d32fc086b47ea8c67589aaeb26b7e07fcbe620d6d0b83e209ea"

def verify_file_sha256(path, expected_sha256):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != expected_sha256:
        raise RuntimeError(f"Security check failed for {path}: unexpected SHA-256")

def load_whisperx_with_verified_vad(model_name, device, vad_options, **model_options):
    from whisperx.vads import pyannote as pyannote_vad
    # Importing lightning.pytorch resets its logger to INFO ("automatically upgraded your checkpoint").
    logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
    vad_path = Path(pyannote_vad.__file__).resolve().parents[1] / "assets" / "pytorch_model.bin"
    verify_file_sha256(vad_path, WHISPERX_VAD_SHA256)

    # WhisperX 3.8.6 bundles this exact legacy checkpoint. Load only that
    # verified file outside the global weights-only policy, then restore it.
    force_weights_only = os.environ.pop("TORCH_FORCE_WEIGHTS_ONLY_LOAD", None)
    try:
        vad_model = pyannote_vad.Pyannote(
            torch.device(device), model_fp=str(vad_path), **vad_options)
    finally:
        if force_weights_only is not None:
            os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = force_weights_only
    # Merge 1 s shorter so padded chunks still fit Whisper's window.
    vad_model.merge_chunks = lambda scores, chunk_size, onset, offset: pad_vad_chunks(split_vad_chunks_at_pauses(
        pyannote_vad.Pyannote.merge_chunks(scores, chunk_size - 1, onset, offset)))

    # load_model keeps vad_options as the thresholds transcribe() merges chunks with.
    # ponytail: swallows its hard-coded "Use manually assigned vad_model" print; errors still raise.
    with contextlib.redirect_stdout(io.StringIO()):
        return whisperx.load_model(
            model_name, device, vad_model=vad_model, vad_options=vad_options, **model_options)

def split_vad_chunks_at_pauses(chunks, pause=2.5, min_turn=0.1):
    """WhisperX glues speech into one chunk across any silence that fits, and Whisper then decodes
    that silence too: it may invent text there, or stop early and skip the line after it. Start a
    new chunk after a pause of 2.5 s or more (WhisperJAV's tuned value for Japanese ASMR-like audio).
    Blips under 0.1 s (clicks, lip smacks) are dropped first, as WhisperX means to but never does,
    so they neither break up a pause nor get decoded alone."""
    split = []
    for chunk in chunks:
        turns, begin = [turn for turn in chunk["segments"] if turn[1] - turn[0] >= min_turn], 0
        for index in range(1, len(turns) + 1):
            if index == len(turns) or turns[index][0] - turns[index - 1][1] >= pause:
                split.append({"start": turns[begin][0], "end": turns[index - 1][1],
                              "segments": turns[begin:index]})
                begin = index
    return split

def pad_vad_chunks(chunks, pad=0.4):
    """WhisperX cuts audio exactly where the VAD score crosses its threshold, clipping soft
    (whispered, breathy) starts and endings. Widen each chunk by up to 0.4 s (faster-whisper's
    speech padding) into the silence around it, sharing a short gap with the neighbour."""
    bounds = [(chunk["start"], chunk["end"]) for chunk in chunks]
    for index, chunk in enumerate(chunks):
        previous_end = bounds[index - 1][1] if index else -math.inf
        next_start = bounds[index + 1][0] if index + 1 < len(bounds) else math.inf
        chunk["start"] = max(0.0, chunk["start"] - pad, (previous_end + chunk["start"]) / 2)
        chunk["end"] = min(chunk["end"] + pad, (chunk["end"] + next_start) / 2)
    return chunks

ANIME_WHISPER_DIR = os.path.join(os.path.dirname(CHECKPOINT_PATH), "_models", "anime-whisper-ct2")

def ensure_anime_whisper(target=ANIME_WHISPER_DIR):
    """Convert litagin/anime-whisper to CTranslate2 once and cache it on Drive."""
    if os.path.isfile(os.path.join(target, "model.bin")):
        return target
    import ctranslate2
    import transformers
    from huggingface_hub import snapshot_download
    print("Converting anime-whisper for WhisperX (first run only)…")
    # ponytail: follows main like kotoba; the Drive cache then freezes it.
    # Safetensors and text configs only: no pickled weights, no custom code.
    source = snapshot_download("litagin/anime-whisper",
                               allow_patterns=["*.json", "*.txt", "*.safetensors"])
    staging = target + ".partial"
    shutil.rmtree(staging, ignore_errors=True)
    ctranslate2.converters.TransformersConverter(
        source, copy_files=["preprocessor_config.json"], load_as_float16=True,
        low_cpu_mem_usage=True).convert(staging, quantization="float16")
    # faster-whisper needs tokenizer.json, which the repo may only ship as vocab/merges.
    transformers.AutoTokenizer.from_pretrained(source).save_pretrained(staging)
    shutil.rmtree(target, ignore_errors=True)
    os.replace(staging, target)
    return target

def format_timestamp(seconds):
    milliseconds = max(0, round(float(seconds) * 1000))
    hours, milliseconds = divmod(milliseconds, 3_600_000)
    minutes, milliseconds = divmod(milliseconds, 60_000)
    secs, milliseconds = divmod(milliseconds, 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{milliseconds:03d}"

def atomic_json_dump(data, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temp_name = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=destination.parent,
                                         prefix=destination.name + ".", suffix=".tmp",
                                         delete=False) as handle:
            temp_name = handle.name
            json.dump(data, handle, ensure_ascii=False,
                      default=lambda value: float(value) if hasattr(value, "__float__") else str(value))
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temp_name, destination)
    except Exception:
        if temp_name:
            Path(temp_name).unlink(missing_ok=True)
        raise

def join_word_texts(words, cjk=False):
    pieces = [str(word.get("word", word.get("text", ""))).strip()
              for word in words]
    pieces = [piece for piece in pieces if piece]
    if cjk:
        return "".join(pieces)
    text = " ".join(pieces)
    return re.sub(r"\s+([,.;:!?])", r"\1", text)

def split_long_segments(segments, max_chars=20, language_code=None, pause_seconds=1.0):
    if language_code not in CJK_LANGUAGE_CODES:
        return segments
    # Kinsoku: never start a line with these; prefer to end one after these.
    NO_LINE_START = set("、。，．・？！?!…」』）)ーぁぃぅぇぉっゃゅょゎァィゥェォッャュョヮ")
    BREAK_AFTER = set("、。，？！?!…")
    def is_timed(word):
        return "start" in word and "end" in word

    output = []
    for segment in segments:
        words = segment.get("words", [])
        if not any(is_timed(word) for word in words):
            output.append(segment)
            continue
        texts = [str(word.get("word", word.get("text", ""))).strip() for word in words]
        def can_start(index):
            return texts[index][:1] not in NO_LINE_START
        def gap_before(index):
            before, after = words[index - 1], words[index]
            return after["start"] - before["end"] if is_timed(before) and is_timed(after) else 0.0
        def word_start(index):  # hiragana -> kanji/katakana usually starts a new word
            before, after = texts[index - 1][-1:], texts[index][:1]
            return "ぁ" <= before <= "ゟ" and not "ぁ" <= after <= "ゟ"

        # Cut at pauses, and when a line gets long cut at the best allowed spot:
        # after punctuation, else a clear pause, else a likely word start, else
        # as late as possible.
        # Untimed words (digits/symbols the aligner skipped) stay in the text.
        pieces, begin = [], 0
        for index in range(1, len(words)):
            if not can_start(index):
                continue
            if gap_before(index) >= pause_seconds:
                pieces.append(words[begin:index]); begin = index
            elif len("".join(texts[begin:index + 1])) > max_chars:
                candidates = [cut for cut in range(begin + 1, index + 1) if can_start(cut)
                              and len("".join(texts[begin:cut])) * 2 >= max_chars] or [index]
                cut = max(candidates, key=lambda cut: (
                    texts[cut - 1][-1:] in BREAK_AFTER, gap_before(cut) >= 0.3,
                    word_start(cut), gap_before(cut), cut))
                pieces.append(words[begin:cut]); begin = cut
        pieces.append(words[begin:])

        # Fold untimed, punctuation-only and tiny pieces into their neighbour
        # unless a pause separates them.
        merged = []
        for chunk in pieces:
            text = join_word_texts(chunk, cjk=True)
            timed = [item for item in chunk if is_timed(item)]
            previous = [item for item in merged[-1] if is_timed(item)] if merged else []
            if merged and (not timed or not text.strip("".join(NO_LINE_START)) or (
                    len(text) <= max_chars // 5 and (not previous
                    or timed[0]["start"] - previous[-1]["end"] < pause_seconds))):
                merged[-1] = merged[-1] + chunk
            else:
                merged.append(chunk)
        if len(merged) > 1 and not any(is_timed(item) for item in merged[0]):
            merged[0:2] = [merged[0] + merged[1]]
        for chunk in merged:
            timed = [item for item in chunk if is_timed(item)]
            output.append({**segment, "start": timed[0]["start"],
                           "end": timed[-1]["end"],
                           "text": join_word_texts(chunk, cjk=True),
                           "words": chunk})
    # Alignment's sentence split can leave a lone "。" or "?" as its own segment.
    folded = []
    for item in output:
        text = str(item.get("text", "")).strip()
        if folded and not text.strip("".join(NO_LINE_START)):
            folded[-1] = {**folded[-1], "text": folded[-1].get("text", "") + text,
                          "words": folded[-1].get("words", []) + item.get("words", [])}
        else:
            folded.append(item)
    return folded

# Stock phrases Whisper tends to invent over silence (learned from video subtitles), in their
# common variants (最後までご視聴いただきありがとうございました). Subtitle credits too.
# ponytail: flagged, not deleted; an outro can really say them.
STOCK_HALLUCINATIONS = re.compile(r"(視聴|ご覧|清聴).{0,8}ありがと|チャンネル登録|字幕|"
                                  r"thanks? (you )?for watching|amara\.org|subtitles by", re.IGNORECASE)

def compression_ratio(text):
    """Whisper's loop detector: a phrase repeated over and over compresses well."""
    data = str(text).strip().encode("utf-8")
    return len(data) / len(zlib.compress(data)) if data else 0.0

def collect_quality_notes(segments, thresholds=None):
    thresholds = thresholds or {"avg_logprob": -1.0, "no_speech_prob": 0.6, "compression_ratio": 2.4}
    notes = []
    for segment in segments:
        reasons = []
        avg_logprob = segment.get("avg_logprob")
        no_speech = segment.get("no_speech_prob")
        compression = segment.get("compression_ratio")
        text = str(segment.get("text", "")).strip()
        if compression is None and text:  # WhisperX reports only avg_logprob
            compression = compression_ratio(text)
        if avg_logprob is not None and avg_logprob < thresholds["avg_logprob"]:
            reasons.append(f"low confidence ({avg_logprob:.2f})")
        if no_speech is not None and no_speech > thresholds["no_speech_prob"]:
            reasons.append(f"possible non-speech ({no_speech:.2f})")
        if compression is not None and compression > thresholds["compression_ratio"]:
            reasons.append(f"repetition risk ({compression:.2f})")
        if segment.get("redecoded"):
            reasons.append("re-decoded after the first pass looped")
        if STOCK_HALLUCINATIONS.search(text):
            reasons.append("stock phrase Whisper often invents over silence")
        if reasons:
            notes.append({"start": segment.get("start"), "end": segment.get("end"),
                          "text": segment.get("text", ""), "reasons": "; ".join(reasons)})
    return notes

def redecode_loops(whisper, segments, audio, threshold=2.4, **options):
    """WhisperX decodes each chunk once, at temperature 0, so a chunk caught in a loop (one phrase
    repeated up to the token limit) loses what was said after the loop began. Decode it again
    as Whisper itself does, sampling at rising temperatures, and keep a result that no longer
    loops. The ladder stops at 0.6 (Japanese tools use 0.3 or less): audio that really repeats
    keeps looping there, and hotter sampling would only trade it for invented text."""
    redone = 0
    for segment in segments:
        if compression_ratio(segment["text"]) <= threshold:
            continue
        clip = audio[int(segment["start"] * 16000):int(segment["end"] * 16000)]
        try:
            pieces, _ = whisper.transcribe(
                clip, temperature=[0.2, 0.4, 0.6], compression_ratio_threshold=threshold,
                log_prob_threshold=None, no_speech_threshold=None, condition_on_previous_text=False,
                without_timestamps=True, **options)
            pieces = list(pieces)  # a chunk fits one 30 s window, so one piece
        except Exception as error:  # a failed retry must not cost the whole file
            print(f"⚠️ Could not re-decode a looping chunk: {error}")
            continue
        text = "".join(piece.text for piece in pieces).strip()
        if text and compression_ratio(text) <= threshold:
            segment.update(text=text, avg_logprob=pieces[0].avg_logprob, redecoded=True)
            redone += 1
    return redone

def collapse_repeats(segments, keep=4, min_repeats=8):
    """Shorten runaway loops (るるる… x80) to a few repeats and an ellipsis."""
    loop = re.compile(r"(.{1,4}?)\1{%d,}" % (min_repeats - 1), re.S)
    for segment in segments:
        segment["text"] = loop.sub(lambda match: match.group(1) * keep + "…", str(segment.get("text", "")))
    return segments

def trim_smeared_words(segments, seconds_per_char=0.5, min_seconds=1.5):
    """The aligner hands the silence after a word (breaths, kisses) to that word, so a
    cue can last 20 s. Starts are reliable; cap how long a word may run."""
    for segment in segments:
        timed = [word for word in segment.get("words", []) if "start" in word and "end" in word]
        for word in timed:
            limit = word["start"] + max(min_seconds, len(str(word.get("word", "")).strip()) * seconds_per_char)
            if word["end"] > limit:
                word["end"] = round(limit, 3)
        if timed:
            segment["end"] = max(word["end"] for word in timed)
    return segments

def widen_crammed_cues(segments, chars_per_second=8.0, max_chars_per_second=16.0):
    """Text the aligner squeezed into a sliver after a silence was usually said during
    it; start such cues earlier, into the free time before them."""
    previous_end = 0.0
    for segment in segments:
        start, end = float(segment.get("start", 0)), float(segment.get("end", 0))
        length = len(str(segment.get("text", "")).strip())
        earlier = max(previous_end, end - length / chars_per_second)
        if length > (end - start) * max_chars_per_second and earlier < start:
            segment["start"] = round(earlier, 3)
        previous_end = max(previous_end, end)
    return segments

def ease_short_cues(segments, chars_per_second=8.0, min_seconds=1.0):
    """Keep cues too brief to read on screen longer, using the silence before the next cue."""
    for index, segment in enumerate(segments):
        start, end = float(segment.get("start", 0)), float(segment.get("end", 0))
        wanted = start + max(min_seconds, len(str(segment.get("text", "")).strip()) / chars_per_second)
        if index + 1 < len(segments):
            wanted = min(wanted, float(segments[index + 1].get("start", wanted)))
        if wanted > end:
            segment["end"] = round(wanted, 3)
    return segments

def find_timing_issues(segments, min_seconds=6.0, min_chars_per_second=1.0,
                       max_chars_per_second=20.0, gap_seconds=60.0):
    """Cues whose timing swallowed silence or is too fast to read, and long stretches with no text."""
    notes = []
    previous_end = 0.0
    for segment in segments:
        text = str(segment.get("text", "")).strip()
        start, end = float(segment.get("start", 0)), float(segment.get("end", 0))
        duration = end - start
        if start - previous_end >= gap_seconds:
            notes.append({"start": previous_end, "end": start, "text": "",
                          "reasons": f"no speech for {(start - previous_end) / 60:.1f} min (missed whispers?)"})
        previous_end = max(previous_end, end)
        if duration >= min_seconds and len(text) < duration * min_chars_per_second:
            reason = f"{len(text)} chars over {duration:.0f}s (timing may be off)"
        elif len(text) > max(duration, 0) * max_chars_per_second:
            reason = f"{len(text)} chars in {duration:.1f}s (too fast to read)"
        else:
            continue
        notes.append({"start": segment.get("start"), "end": segment.get("end"), "text": text, "reasons": reason})
    return notes

def merge_api_segments(chunk_results):
    merged = []
    detected_language = None
    duration = 0.0
    for chunk in chunk_results:
        offset = float(chunk["offset_seconds"])
        response = chunk["response"]
        detected_language = detected_language or response.get("language")
        for original in response.get("segments") or []:
            segment = dict(original)
            segment["start"] = float(segment.get("start") or 0) + offset
            segment["end"] = float(segment.get("end") or 0) + offset
            merged.append(segment)
            duration = max(duration, segment["end"])
    merged.sort(key=lambda segment: (segment["start"], segment["end"]))
    for index, segment in enumerate(merged):
        segment["id"] = index  # chunk-local ids restart at 0; keep them unique
    return {"segments": merged, "language": detected_language, "duration": duration}

def prepare_api_chunks(audio_path, temp_dir, max_bytes=24 * 1024 * 1024):
    supported = {"flac", "mp3", "mp4", "mpeg", "mpga", "m4a", "ogg", "wav", "webm"}
    extension = Path(audio_path).suffix.lower().lstrip(".")
    if extension in supported and os.path.getsize(audio_path) < max_bytes:
        return [{"path": audio_path, "offset_seconds": 0.0}]

    probe = subprocess.run([
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", audio_path,
    ], check=True, capture_output=True, text=True)
    try:
        duration_seconds = float(probe.stdout.strip())
    except ValueError:
        duration_seconds = 0.0
    if not math.isfinite(duration_seconds) or duration_seconds <= 0:
        raise RuntimeError(f"Could not read a valid duration for {audio_path}; is it an audio/video file?")
    # 64 kbps CBR plus 15% container/headroom; the size check below is authoritative.
    target_seconds = max(1.0, max_bytes * 8 / 64_000 * 0.85)
    chunks = []

    def export_piece(start_seconds, piece_seconds):
        path = os.path.join(temp_dir, f"chunk-{len(chunks):04d}.mp3")
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start_seconds), "-i", audio_path,
            "-t", str(piece_seconds), "-vn", "-ac", "1", "-ar", "16000",
            "-b:a", "64k", path,
        ], check=True, capture_output=True)
        if os.path.getsize(path) >= max_bytes:
            os.remove(path)
            if piece_seconds <= 1.0:
                raise RuntimeError("Unable to create an API chunk below the upload limit")
            first = piece_seconds / 2
            export_piece(start_seconds, first)
            export_piece(start_seconds + first, piece_seconds - first)
        else:
            chunks.append({"path": path, "offset_seconds": float(start_seconds)})

    # Equal-length pieces avoid a float-drift sliver chunk at the very end.
    piece_count = math.ceil(duration_seconds / target_seconds)
    piece_seconds = duration_seconds / piece_count
    for index in range(piece_count):
        export_piece(index * piece_seconds, piece_seconds)
    if any(os.path.getsize(chunk["path"]) >= max_bytes for chunk in chunks):
        raise RuntimeError("API chunk verification failed")
    return chunks

def free_vram():
    gc.collect()
    if "torch" in globals() and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def is_cuda_out_of_memory(error):
    message = str(error).lower()
    return "out of memory" in message and ("cuda" in message or "cublas" in message)

def transcribe_with_batch_backoff(model, audio, initial_batch_size, **options):
    current_batch_size = initial_batch_size
    while True:
        try:
            return (model.transcribe(audio, batch_size=current_batch_size, **options),
                    current_batch_size)
        except RuntimeError as error:
            if not is_cuda_out_of_memory(error):
                raise
            if current_batch_size == 1:
                raise RuntimeError(
                    "CUDA ran out of memory even at batch_size=1. Restart the runtime, "
                    "then use a smaller model or compute_type='int8'.") from error
            next_batch_size = max(1, current_batch_size // 2)
            print(f"⚠️ CUDA out of memory at batch_size={current_batch_size}; "
                  f"retrying with batch_size={next_batch_size}…")
            current_batch_size = next_batch_size
        # Run after leaving the exception handler so its traceback no longer retains tensors.
        free_vram()

SAFE_ALIGN_MODELS_HF = {
    # Immutable revisions verified to contain safetensors weights.
    "ja": ("jonatasgrosman/wav2vec2-large-xlsr-53-japanese", "2785e99ab97df77a32b5bd0ece5c9fa188a02f19"),
    "cs": ("comodoro/wav2vec2-xls-r-300m-cs-250", "73e2b9004f35d3ca79316a624e7edc3cdaa45d40"),
    "da": ("saattrupdan/wav2vec2-xls-r-300m-ftspeech", "7a60985d596e7a9e26b01a0707fe59daac6ffe84"),
    "eu": ("stefan-it/wav2vec2-large-xlsr-53-basque", "ac4e27d2d376a5ed919b381ec99d7a86b37c0a6b"),
    "he": ("imvladikon/wav2vec2-xls-r-300m-hebrew", "b2e683004903f1b11bee2c6092a7c323ed858d82"),
    "hi": ("theainerd/Wav2Vec2-large-xlsr-hindi", "062f7f566e2671336992b011dcb9387cd3cffe5e"),
    "hr": ("classla/wav2vec2-xls-r-parlaspeech-hr", "a6075e0c9e0c49a533dc61876b1eafc6bc78de92"),
    "ko": ("kresnik/wav2vec2-large-xlsr-korean", "629c9a3501c10ba128bf3fa1eebb12af3be03f61"),
    "lv": ("jimregan/wav2vec2-large-xlsr-latvian-cv", "ca99957784deb8ffadf4b782db3e96dde797d164"),
    "nn": ("NbAiLab/nb-wav2vec2-1b-nynorsk", "37107327f6af742c008042d8cfdba4276de18f82"),
    "no": ("NbAiLab/nb-wav2vec2-1b-bokmaal-v2", "58a6eb5d7927c378f5d2785bc865756c9ae273ae"),
    "ro": ("gigant/romanian-wav2vec2", "79cf603aac59501d02bfeb37f615efc3ac4ce1b3"),
    "sv": ("KBLab/wav2vec2-large-voxrex-swedish", "ca70e31c06a2617bf7fe3b4fb5d387d2b19b2983"),
    "uk": ("Yehor/wav2vec2-xls-r-300m-uk-with-small-lm", "e3ced4def0d70be3aab0f2db598a59961fe9ab3b"),
    "ur": ("kingabzpro/wav2vec2-large-xls-r-300m-Urdu", "18a6595fc0cc4d9c63fc8d0407b5596f09c49edb"),
}

def load_safe_align_model(language_code, device):
    from whisperx.alignment import DEFAULT_ALIGN_MODELS_TORCH
    if language_code in DEFAULT_ALIGN_MODELS_TORCH:
        model_a, metadata = whisperx.load_align_model(language_code=language_code, device=device)
        return model_a, metadata, {
            "source": "torchaudio", "model": DEFAULT_ALIGN_MODELS_TORCH[language_code],
            "revision": "torchaudio==2.8.0+cu128", "weights_only": True}

    safe_model = SAFE_ALIGN_MODELS_HF.get(language_code)
    if not safe_model:
        raise RuntimeError(
            f"No pinned safetensors alignment model is available for language '{language_code}'; "
            "unsafe pickle fallback is disabled")
    model_id, revision = safe_model
    from huggingface_hub import snapshot_download
    safe_model_dir = snapshot_download(
        model_id, revision=revision,
        allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"],
    )
    if not list(Path(safe_model_dir).glob("*.safetensors")):
        raise RuntimeError(f"Pinned alignment model {model_id}@{revision} did not provide safetensors")
    model_a, metadata = whisperx.load_align_model(
        language_code=language_code, device=device, model_name=safe_model_dir)
    return model_a, metadata, {
        "source": "huggingface", "model": model_id,
        "revision": revision, "weights": "safetensors"}

use_openai = transcription_backend.startswith("OpenAI")
if use_openai:
    print("🔒 OpenAI mode selected: audio will be uploaded to OpenAI for processing.")
    try:
        openai_api_key = userdata.get("OPENAI_API_KEY") or ""
    except Exception:
        openai_api_key = ""
    if not openai_api_key:
        raise RuntimeError("Add OPENAI_API_KEY in Colab Secrets and enable notebook access")
    from openai import OpenAI
    api_client = OpenAI(api_key=openai_api_key)
else:
    print("🔒 Local mode selected: audio remains in this Colab runtime.")
    for logger_name in ("pytorch_lightning", "lightning_fabric", "pyannote", "speechbrain"):
        logging.getLogger(logger_name).setLevel(logging.ERROR)
    for warning_text in (".*TensorFloat-32.*", ".*was trained with.*", ".*does not match the version.*",
                         ".*gradient_checkpointing.*"):
        warnings.filterwarnings("ignore", message=warning_text)
    try:
        hf_token = userdata.get("HF_TOKEN") or ""
    except Exception:
        hf_token = ""
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("Hugging Face token loaded ✅")
    elif word_timestamps:
        print("ℹ️ No HF_TOKEN secret found. Public alignment models may still work; gated models will not.")

    import torch
    import whisperx
    whisperx.setup_logging("warning")  # hide per-file INFO lines
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":
        compute_type = "int8"
        print("⚠️ GPU unavailable; using CPU/int8. This can be very slow.")
    else:
        print("Using GPU")
        subprocess.run(["nvidia-smi", "-L"], check=False)
        print(f"torch CUDA devices: {torch.cuda.device_count()} | "
              f"{torch.cuda.get_device_name(0) if torch.cuda.device_count() else 'none'}")
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    if use_model == "anime-whisper":
        use_model = ensure_anime_whisper()
    model_name = use_model
    # WhisperX's batched decoder uses only beam search options: temperature fallback,
    # previous-text conditioning and no-speech/compression thresholds are ignored.
    asr_options = {
        "beam_size": quality["beam_size"],
        "hotwords": effective_prompt or None,
        "no_repeat_ngram_size": 10 if reduce_repetition else 0,
    }
    # Pyannote defaults, or a lower threshold that keeps whispered speech. 15 s chunks for
    # whispers and for the Kotoba-based models, which learned from short clips (as Kotoba runs them).
    vad_options = {"chunk_size": 15 if quiet_speech or kotoba_based else 30,
                   "vad_onset": 0.3 if quiet_speech else 0.5,
                   "vad_offset": 0.2 if quiet_speech else 0.363}
    # asr_options are baked in at load time, so any change must force a reload.
    model_signature = (model_name, DEVICE, compute_type, lang_code,
                       json.dumps([asr_options, vad_options], sort_keys=True))
    if "model" in globals() and globals().get("_model_signature") == model_signature:
        print(f"Reusing {model_name} already in memory ⚡")
    else:
        if "model" in globals():
            del model
        free_vram()
        print(f"Loading {model_name} with WhisperX…")
        model = load_whisperx_with_verified_vad(
            model_name, DEVICE, vad_options, compute_type=compute_type, language=lang_code,
            asr_options=asr_options,
        )
        _model_signature = model_signature

results = {}
alignment_failures = []
print("-- TRANSLATE TO ENGLISH --" if task == "translate" else "-- TRANSCRIPTION --")

for audio_path in audio_files:
    print(f"\nProcessing: {audio_path}")
    if use_openai:
        api_options = {"response_format": "verbose_json"}
        if effective_prompt:
            api_options["prompt"] = effective_prompt
        if lang_code and task == "transcribe":
            api_options["language"] = lang_code
        endpoint = (api_client.audio.transcriptions if task == "transcribe"
                    else api_client.audio.translations).create
        chunk_results = []
        with tempfile.TemporaryDirectory(prefix="audio-to-text-") as temp_dir:
            chunks = prepare_api_chunks(audio_path, temp_dir)
            for index, chunk in enumerate(chunks, 1):
                print(f"Uploading chunk {index}/{len(chunks)} ({chunk['offset_seconds']:.1f}s offset)…")
                with open(chunk["path"], "rb") as audio_handle:
                    response = endpoint(model="whisper-1", file=audio_handle, **api_options).model_dump()
                chunk_results.append({"offset_seconds": chunk["offset_seconds"], "response": response})
        result = merge_api_segments(chunk_results)
        if task == "translate":
            result["source_language"] = result.get("language")
            result["language"] = "en"
        raw_asr_segments = result.get("segments", [])
        result["quality_notes"] = collect_quality_notes(raw_asr_segments)
        if reduce_repetition:
            collapse_repeats(raw_asr_segments)
        result["alignment_status"] = "not_available_for_openai_backend"
    else:
        audio = whisperx.load_audio(audio_path)
        audio_seconds = len(audio) / 16000
        print(f"Transcribing ~{audio_seconds / 60:.1f} min (batch_size={batch_size})…")
        try:
            result, used_batch_size = transcribe_with_batch_backoff(
                model, audio, batch_size, language=lang_code, task=task,
                chunk_size=vad_options["chunk_size"], print_progress=False)
            if used_batch_size != batch_size:
                print(f"Continuing with batch_size={used_batch_size}.")
                batch_size = used_batch_size
        except RuntimeError as error:
            if "cudaErrorInvalidDevice" in str(error) or "invalid device ordinal" in str(error):
                raise RuntimeError(
                    "The GPU/device is invalid (cudaErrorInvalidDevice). Colab may have "
                    "reclaimed or reset the GPU, or the cached model is bound to a dead "
                    "device. Fix: Runtime \u2192 Restart runtime, then re-run from Step 1. If it "
                    "persists, Runtime \u2192 Change runtime type to a different GPU (or Factory "
                    "reset the runtime). You can also try compute_type='int8'."
                ) from error
            raise
        detected_language = result.get("language", lang_code)
        redone = redecode_loops(model.model, result.get("segments", []), audio, language=detected_language,
                                task=task, hotwords=asr_options["hotwords"],
                                no_repeat_ngram_size=asr_options["no_repeat_ngram_size"])
        if redone:
            print(f"↻ Re-decoded {redone} chunk(s) caught in a repetition loop.")
        # anime-whisper answers a lone breath with "…"; a segment without a word is noise.
        raw_asr_segments = result["segments"] = [
            segment for segment in result.get("segments", []) if re.search(r"\w", segment.get("text", ""))]
        for segment in raw_asr_segments:  # VAD padding can run past the end of the file
            segment["end"] = min(segment["end"], round(audio_seconds, 3))
        raw_quality_notes = collect_quality_notes(raw_asr_segments)
        if reduce_repetition:
            collapse_repeats(raw_asr_segments)  # before alignment, which rebuilds text from words
        result["alignment_status"] = "not_requested"
        if word_timestamps and task == "transcribe":
            try:
                print("Aligning word timestamps…")
                model_a, metadata, alignment_model = load_safe_align_model(
                    detected_language, DEVICE)
                result = whisperx.align(result["segments"], model_a, metadata,
                                        audio, DEVICE, return_char_alignments=False)
                result["language"] = detected_language
                result["alignment_status"] = "complete"
                result["alignment_model"] = alignment_model
                del model_a, metadata
                free_vram()
            except Exception as error:
                result["alignment_status"] = "failed"
                result["alignment_error"] = str(error)
                alignment_failures.append(f"{audio_path}: {error}")
                print(f"⚠️ Word alignment failed: {error}")
                if fail_if_alignment_fails:
                    raise RuntimeError(f"Required alignment failed for {audio_path}: {error}") from error
        elif task == "translate":
            result["alignment_status"] = "not_applicable_to_translation"
        result["segments"] = split_long_segments(
            trim_smeared_words(result["segments"]), max_chars=max_chars_per_line,
            language_code=detected_language)
        result.setdefault("language", detected_language)
        result["quality_notes"] = raw_quality_notes
        if task == "translate":
            result["source_language"] = detected_language
            result["language"] = "en"
        del audio
        free_vram()

    ease_short_cues(widen_crammed_cues(result.get("segments", [])))
    for segment in result.get("segments", []):
        segment["text"] = segment.get("text", "").strip()
        print(f"[{format_timestamp(segment.get('start', 0))} --> {format_timestamp(segment.get('end', 0))}] {segment['text']}")
    result["text"] = "\n".join(segment["text"] for segment in result.get("segments", []))
    notes_by_start = {}  # one line per segment, even when several checks flag it
    for note in result.get("quality_notes", []) + find_timing_issues(result.get("segments", [])):
        start = note.get("start") or 0
        if start in notes_by_start:
            notes_by_start[start]["reasons"] += "; " + note["reasons"]
        else:
            notes_by_start[start] = dict(note)
    result["quality_notes"] = [notes_by_start[start] for start in sorted(notes_by_start)]
    results[audio_path] = result
    if result["quality_notes"]:
        print(f"  ⚠️ Review {len(result['quality_notes'])} spot(s): {audio_path}")
        for note in result["quality_notes"]:
            text = f": {note['text'][:30]}" if note["text"] else ""
            print(f"    [{format_timestamp(note.get('start') or 0)}] {note['reasons']}{text}")
    checkpoint = {
        "schema_version": 1,
        "task": task,
        "backend": transcription_backend,
        "inputs": audio_files,
        "results": results,
    }
    atomic_json_dump(checkpoint, CHECKPOINT_PATH)
    print(f"💾 Checkpoint updated: {CHECKPOINT_PATH}")

print(f"\n✅ Done. Processed {len(results)} file(s).")
if alignment_failures:
    print("⚠️ Completed with alignment degradation:")
    for failure in alignment_failures:
        print("  •", failure)
print("Run Step 4 to write outputs.")


## 4️⃣ 💾 Save your results

Outputs are saved persistently to **Google Drive** by default. If the runtime lost its in-memory state, this step validates and loads the last Step 3 checkpoint automatically.

Repetition cleanup is off by default because repetition can be legitimate speech. Raw JSON keeps the uncleaned word-level data beside the cleaned outputs.


In [ ]:
#@title 💾 Step 4 — Save results { display-mode: "form" }
import copy
import hashlib
import json
import os
import re
import tempfile
from pathlib import Path
from typing import TextIO

output_dir = "/content/drive/MyDrive/audio_transcription" #@param {type:"string"}
output_formats = "srt" #@param ["txt,vtt,srt,tsv,json", "txt,vtt,srt", "txt,srt", "txt", "vtt", "srt", "tsv", "json"] {allow-input: true}
cleanup_repetitions = False #@param {type:"boolean"}
save_raw_json = True #@param {type:"boolean"}
overwrite_existing = False #@param {type:"boolean"}

ALLOWED_OUTPUT_FORMATS = {"txt", "vtt", "srt", "tsv", "json"}
CHECKPOINT_PATH = "/content/drive/MyDrive/audio_transcription/_last_results.json"

def validate_output_formats(value):
    formats = [item.strip().lower() for item in value.split(",") if item.strip()]
    invalid = sorted(set(formats) - ALLOWED_OUTPUT_FORMATS)
    if not formats or invalid:
        raise ValueError(f"Output formats must be from {sorted(ALLOWED_OUTPUT_FORMATS)}; invalid: {invalid}")
    return list(dict.fromkeys(formats))

def load_results_checkpoint(path):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("schema_version") != 1 or not isinstance(payload.get("results"), dict):
        raise ValueError("Unsupported or invalid results checkpoint")
    return payload

def clean_repeated_words(text, threshold=4):
    if not text:
        return text
    # A loop collapses to one copy; a short sound keeps three, so a laugh stays one (ははは).
    text = re.sub(r"(.{1,10}?)\1{3,}", lambda match: match.group(1) * (
        3 if len(match.group(1)) <= 3 and match.group(1).isalnum() else 1), text)
    text = re.sub(r"\b(\w+)(\s+\1){" + str(threshold - 1) + r",}\b",
                  r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"(.{8,60}?)(\s*\1){" + str(threshold - 2) + r",}",
                  r"\1", text, flags=re.IGNORECASE)
    text = re.sub(r"(\.\.\.)(\s*\.\.\.)+", r"\1", text)
    return re.sub(r"\s{2,}", " ", text).strip()

def clean_segments(segments):
    cleaned = []
    seen = []
    for segment in segments:
        raw_text = segment.get("text", "").strip()
        stretch_text = raw_text.rstrip("\ufffd")
        text = clean_repeated_words(raw_text)
        stretched_continuation = (
            len(stretch_text) >= 10 and len(set(stretch_text)) == 1 and
            any(previous.endswith(stretch_text[0]) for previous in seen[-2:]))
        duplicate = stretched_continuation or any(text and previous and (
            text == previous or
            (len(text) > 10 and text in previous) or
            (len(previous) > 10 and previous in text)
        ) for previous in seen[-2:])
        if text and not duplicate:
            seen.append(text)
            cleaned.append({**segment, "text": text})
    return cleaned

def output_name_map(paths):
    stems = [Path(path).stem for path in paths]
    counts = {stem: stems.count(stem) for stem in set(stems)}
    return {
        path: (stem if counts[stem] == 1 else
               f"{stem}-{hashlib.sha256(path.encode('utf-8')).hexdigest()[:8]}")
        for path, stem in zip(paths, stems)
    }

def reserve_output_name(base_name, formats, directory, overwrite=False):
    if overwrite:
        return base_name
    suffix = 0
    while True:
        candidate = base_name if suffix == 0 else f"{base_name}-{suffix}"
        reserved = []
        try:
            for extension in formats:
                destination = os.path.join(directory, f"{candidate}.{extension}")
                descriptor = os.open(destination, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                os.close(descriptor)
                reserved.append(destination)
            return candidate
        except FileExistsError:
            for destination in reserved:
                os.unlink(destination)
            suffix += 1
        except Exception:
            for destination in reserved:
                os.unlink(destination)
            raise

try:
    results
except NameError:
    results = None
if not results:
    if not os.path.isfile(CHECKPOINT_PATH):
        raise RuntimeError("No in-memory results or checkpoint found. Run Step 3 first.")
    checkpoint_payload = load_results_checkpoint(CHECKPOINT_PATH)
    results = checkpoint_payload["results"]
    task = checkpoint_payload.get("task", "transcribe")
    print(f"♻️ Recovered {len(results)} file(s) from {CHECKPOINT_PATH}")

output_formats = validate_output_formats(output_formats)
os.makedirs(output_dir, exist_ok=True)
from whisperx.utils import get_writer, WriteTXT

class WriteText(WriteTXT):
    def write_result(self, result: dict, file: TextIO, **kwargs):
        print(result["text"], file=file, flush=True)

def write_result(result, output_format, output_file_name, raw=False):
    safe_result = copy.deepcopy(result)
    if cleanup_repetitions and not raw:
        safe_result["segments"] = clean_segments(safe_result.get("segments", []))
        safe_result["text"] = "\n".join(segment["text"] for segment in safe_result["segments"])

    destination = os.path.join(output_dir, f"{output_file_name}.{output_format}")
    if output_format == "json":
        temp_path = None
        try:
            with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=output_dir,
                                             suffix=".tmp", delete=False) as handle:
                temp_path = handle.name
                json.dump(safe_result, handle, ensure_ascii=False, indent=2,
                          default=lambda value: float(value) if hasattr(value, "__float__") else str(value))
                handle.flush()
                os.fsync(handle.fileno())
            os.replace(temp_path, destination)
        except Exception:
            if temp_path:
                Path(temp_path).unlink(missing_ok=True)
            raise
        return destination

    for segment in safe_result.get("segments", []):
        segment.pop("words", None)
        if output_format == "tsv":  # one row per segment
            segment["text"] = " ".join(str(segment.get("text", "")).split())
    writer_options = {"max_line_width": None, "max_line_count": None,
                      "highlight_words": False}
    with tempfile.TemporaryDirectory(dir=output_dir) as temp_dir:
        writer = WriteText(temp_dir) if output_format == "txt" else get_writer(output_format, temp_dir)
        # WhisperX writers strip everything after the last dot of this name, so use a
        # fixed placeholder; "my.song" would otherwise be written as "my.srt".
        writer(safe_result, "output", writer_options)
        generated = os.path.join(temp_dir, f"output.{output_format}")
        os.replace(generated, destination)
    return destination

def normalize_subtitle(path):
    path = Path(path)
    raw = path.read_bytes()
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]
    raw = raw.replace(b"\r\n", b"\n").replace(b"\r", b"\n")
    normalized = b"\xef\xbb\xbf" + raw.replace(b"\n", b"\r\n")
    with tempfile.NamedTemporaryFile("wb", dir=path.parent, suffix=".tmp", delete=False) as handle:
        temporary = handle.name
        handle.write(normalized)
    try:
        os.replace(temporary, path)
    except Exception:
        Path(temporary).unlink(missing_ok=True)
        raise

name_map = output_name_map(list(results))
generated_paths = []
for audio_path, result in results.items():
    reserved_formats = list(output_formats)
    if save_raw_json:
        reserved_formats.append("raw.json")
    base_name = reserve_output_name(name_map[audio_path], reserved_formats, output_dir,
                                    overwrite=overwrite_existing)
    reservations = [os.path.join(output_dir, f"{base_name}.{extension}")
                    for extension in reserved_formats]
    try:
        for output_format in output_formats:
            path = write_result(result, output_format, base_name)
            generated_paths.append(path)
            print("✅", path)
        if save_raw_json:
            raw_path = write_result(result, "json", base_name + ".raw", raw=True)
            generated_paths.append(raw_path)
            print("✅", raw_path)
    except Exception:
        # Only remove our empty placeholders; with overwrite on these are real old outputs.
        if not overwrite_existing:
            for reservation in reservations:
                if reservation not in generated_paths:
                    Path(reservation).unlink(missing_ok=True)
        raise

for path in generated_paths:
    if Path(path).suffix.lower() in {".srt", ".vtt"}:
        normalize_subtitle(path)
print(f"\n✅ Saved {len(generated_paths)} file(s) persistently in {output_dir}")


## 5️⃣ 💬 Translate with DeepL *(optional)*

This step sends transcript text—not audio—to DeepL after you explicitly run it. Add `DEEPL_API_KEY` in Colab Secrets first. Translation progress is checkpointed in Drive after every batch.

Translated files are saved in the Step 4 output folder.


In [ ]:
#@title 💬 Step 5 — Translate with DeepL { display-mode: "form" }
import hashlib
import json
import os
import re
import tempfile
from pathlib import Path
import deepl
from google.colab import userdata

#@markdown Transcript text is sent to DeepL only when this cell is run. Short Step 3
#@markdown display fragments are re-joined into whole sentences before translation.
deepl_target_language = "English (American)" #@param ["Bulgarian", "Chinese (simplified)", "Czech", "Danish", "Dutch", "English (American)", "English (British)", "Estonian", "Finnish", "French", "German", "Greek", "Hungarian", "Indonesian", "Italian", "Japanese", "Korean", "Latvian", "Lithuanian", "Norwegian", "Polish", "Portuguese (Brazilian)", "Portuguese (European)", "Romanian", "Russian", "Slovak", "Slovenian", "Spanish", "Swedish", "Turkish", "Ukrainian"]
deepl_formality = "default" #@param ["default", "formal", "informal"]
share_context = True #@param {type:"boolean"}
#@markdown Style hint for DeepL (EN/DE/ES/FR/IT/JA/KO/ZH targets; ignored if your plan rejects it). Clear it to turn off.
deepl_instructions = "These are spoken lines from an audio recording. Translate them as natural, concise subtitles, and render sound effects (breaths, kisses, laughter) as short, consistent interjections." #@param {type:"string"}
# Bump whenever the translation scheme changes so stale checkpoints are not reused.
TRANSLATION_VERSION = 7

if "write_result" not in globals() or "output_formats" not in globals():
    raise RuntimeError("Run Step 4 once before Step 5 to choose output settings")
try:
    deepl_api_key = userdata.get("DEEPL_API_KEY") or ""
except Exception:
    deepl_api_key = ""
if not deepl_api_key:
    raise RuntimeError("Add DEEPL_API_KEY in Colab Secrets and enable notebook access")

if "results" not in globals() or not results:
    checkpoint_payload = load_results_checkpoint(CHECKPOINT_PATH)
    results = checkpoint_payload["results"]
    task = checkpoint_payload.get("task", "transcribe")

DEEPL_CHECKPOINT = os.path.join(output_dir, "_last_deepl_results.json")

def atomic_deepl_checkpoint(data):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=output_dir,
                                         suffix=".tmp", delete=False) as handle:
            temporary = handle.name
            json.dump({"schema_version": 1, "translation_version": TRANSLATION_VERSION,
                       "target_code": target_code,
                       "source_signature": source_signature, "results": data},
                      handle, ensure_ascii=False, indent=2)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, DEEPL_CHECKPOINT)
    except Exception:
        if temporary:
            Path(temporary).unlink(missing_ok=True)
        raise

def valid_resume_prefix(saved_segments, source_segments):
    if not isinstance(saved_segments, list) or len(saved_segments) > len(source_segments):
        return False
    for saved, source in zip(saved_segments, source_segments):
        if saved.get("id") != source.get("id"):
            return False
        if float(saved.get("start", -1)) != float(source.get("start", -2)):
            return False
        if float(saved.get("end", -1)) != float(source.get("end", -2)):
            return False
    return True

# WhisperX leaves stray U+FFFD characters and fragments that begin/end on a
# Japanese comma; DeepL turns those into ", right?" noise.
def normalize_translation_source(text):
    if not text:
        return ""
    boundary = "、。・…"
    text = text.replace("\ufffd", "")
    text = re.sub("^[" + boundary + r"\s]+", "", text)
    text = re.sub("[" + boundary + r"\s]+$", "", text)
    return text.strip()

# Step 3 emits ~20-char display fragments whose sentence ends usually fall
# mid-fragment. Flatten fragments to per-character samples with interpolated
# timestamps, then cut a unit at a sentence end, a no-speech gap, or a hard
# time/length cap. Sentence ends inside the first min_seconds are ignored, so
# "ここは?すりすり…ここは?" becomes one readable cue instead of four 0.1 s flashes.
def build_translation_units(segments, max_seconds=12.0, max_chars=120,
                            gap_seconds=1.5, min_seconds=1.5):
    sentence_end = "。！？.?!…"
    closers = "」』）)\"'”’"
    units, current = [], []

    def flush():
        text = normalize_translation_source("".join(s["char"] for s in current))
        if text:
            units.append({"id": current[0]["id"], "start": current[0]["start"],
                          "end": current[-1]["end"], "text": text})
        current.clear()

    def ends_sentence():
        index = len(current) - 1
        while index >= 0 and current[index]["char"] in closers:
            index -= 1
        return index >= 0 and current[index]["char"] in sentence_end

    for segment in segments:
        raw = str(segment.get("text", "")).replace("\ufffd", "")
        if not raw.strip():
            continue
        try:
            seg_start = float(segment["start"])
            seg_end = max(float(segment["end"]), seg_start)
        except (KeyError, TypeError, ValueError):
            continue
        step = (seg_end - seg_start) / len(raw)
        for index, char in enumerate(raw):
            sample = {"char": char, "start": seg_start + step * index,
                      "end": seg_start + step * (index + 1), "id": segment.get("id")}
            # Cut after a run like "?!」" or "...", but a Latin "." needs a following
            # space so "3.5" and "e.g" are not treated as sentence ends.
            if (current and char not in sentence_end and char not in closers
                    and current[-1]["end"] - current[0]["start"] >= min_seconds
                    and ends_sentence()
                    and (current[-1]["char"] != "." or char.isspace())):
                flush()
            if current and (sample["start"] - current[-1]["end"] > gap_seconds
                            or sample["end"] - current[0]["start"] > max_seconds
                            or len(current) >= max_chars):
                flush()
            current.append(sample)
    flush()
    return units

def split_display_lines(text, max_len=42):
    """Break a translated sentence into short, readable subtitle lines."""
    if not text or len(text) <= max_len:
        return text
    tokens = []
    for token in re.split(r"(\s+)", text):
        # Hard-wrap tokens with no spaces (CJK targets) instead of one huge line.
        tokens.extend(token[i:i + max_len] for i in range(0, len(token), max_len))
    lines, current = [], ""
    for token in tokens:
        if current and len(current) + len(token) > max_len:
            lines.append(current.strip())
            current = token
        else:
            current += token
    if current.strip():
        lines.append(current.strip())
    return "\n".join(line for line in lines if line)

def collapse_translation_repeats(text, keep=3):
    """Shorten DeepL's walls of repeats ("slurp-slurp-slurp-…", "rurururu…")."""
    text = re.sub(r"\b(\w+)([\s,\-–—]+\1\b){%d,}" % (keep + 1),
                  lambda match: match.group(1) + match.group(2) * (keep - 1) + "…",
                  text, flags=re.IGNORECASE)
    def shorten(match):  # keep a trailing partial repeat ("…u") only if it is real text
        unit, tail = match.group(1), match.group(2)
        return unit * 4 + "…" + ("" if unit.startswith(tail) else tail)
    return re.sub(r"(\w{1,4}?)\1{7,}(\w{0,3})", shorten, text)

def subtitle_segments(segments, max_lines=2, chars_per_second=15.0, min_seconds=1.0):
    """Wrap translations into cues of at most max_lines lines, sharing the unit's time
    by length, and keep brief cues on screen long enough to read."""
    cues = []
    for segment in segments:
        # One line of text in: a stray newline would hide everything before it from the split below.
        text = collapse_translation_repeats(" ".join(str(segment.get("text", "")).split()))
        if not text:
            continue
        # Pack whole sentences into each cue; only a sentence too long on its own
        # is cut between lines.
        sentences = re.findall(r".+?(?:[.!?…]+[\"'”’)]*(?:\s+|$)|[。！？]+[」』）]*|$)", text)
        chunks, current = [], ""
        for sentence in sentences:
            if current and split_display_lines((current + sentence).strip()).count("\n") >= max_lines:
                chunks.append(current)
                current = ""
            current += sentence
        chunks.append(current)
        lines_per_chunk = [split_display_lines(chunk.strip()).split("\n") for chunk in chunks]
        chunks = ["\n".join(lines[i:i + max_lines]) for lines in lines_per_chunk
                  for i in range(0, len(lines), max_lines)]
        start, end = float(segment["start"]), float(segment["end"])
        total = sum(len(chunk) for chunk in chunks) or 1
        for number, chunk in enumerate(chunks, 1):
            chunk_end = end if number == len(chunks) else start + (end - start) * len(chunk) / total
            cues.append({"id": segment.get("id"), "start": round(start, 3),
                         "end": round(chunk_end, 3), "text": chunk})
            start = chunk_end
    for index, cue in enumerate(cues):
        wanted = cue["start"] + max(min_seconds, len(cue["text"]) / chars_per_second)
        if index + 1 < len(cues):
            wanted = min(wanted, cues[index + 1]["start"])
        cue["end"] = round(max(cue["end"], wanted), 3)
    return cues

def deepl_source_code(language):
    from whisperx.utils import TO_LANGUAGE_CODE
    code = str(language or "").lower()
    code = TO_LANGUAGE_CODE.get(code, code)  # OpenAI returns names such as "japanese"
    return {"no": "NB", "nn": "NB"}.get(code, code.upper())

translator = deepl.Translator(deepl_api_key)
targets = translator.get_target_languages()
target = next((item for item in targets if item.name == deepl_target_language), None)
if not target:
    raise ValueError(f"DeepL target is unavailable: {deepl_target_language}")
target_code = target.code.upper()
formality = None if deepl_formality == "default" else (
    "prefer_more" if deepl_formality == "formal" else "prefer_less")
source_codes = {item.code.upper() for item in translator.get_source_languages()}
translation_signature_payload = {
    "version": TRANSLATION_VERSION,
    "formality": formality,
    "share_context": bool(share_context),
    "cleanup_repetitions": bool(cleanup_repetitions),
    "instructions": deepl_instructions.strip(),
}
# DeepL accepts up to 10 instructions of 300 chars, only for these targets.
custom_instructions = ([deepl_instructions.strip()[:300]] if deepl_instructions.strip()
                       and target_code.split("-")[0] in {"DE", "EN", "ES", "FR", "IT", "JA", "KO", "ZH"}
                       else None)
source_signature = hashlib.sha256(
    json.dumps({"results": results, "translation": translation_signature_payload},
               ensure_ascii=False, sort_keys=True, default=str).encode("utf-8")
).hexdigest()
translated_results = {}
if os.path.isfile(DEEPL_CHECKPOINT):
    try:
        with open(DEEPL_CHECKPOINT, encoding="utf-8") as handle:
            saved_translation = json.load(handle)
        compatible = (
            saved_translation.get("schema_version") == 1
            and saved_translation.get("translation_version") == TRANSLATION_VERSION
            and saved_translation.get("target_code") == target_code
            and saved_translation.get("source_signature") == source_signature
        )
        if compatible:
            translated_results = saved_translation.get("results", {})
            print(f"♻️ Loaded compatible DeepL checkpoint: {DEEPL_CHECKPOINT}")
        else:
            print("ℹ️ Existing DeepL checkpoint belongs to different inputs/settings; starting fresh.")
    except (OSError, ValueError, TypeError, AttributeError) as error:
        print(f"⚠️ Ignoring invalid DeepL checkpoint: {error}")

print("🔒 Sending transcript text to DeepL. Audio is not uploaded.")
for audio_path, result in results.items():
    existing = translated_results.get(audio_path, {})
    if existing.get("translation_status") == "complete":
        print(f"♻️ Reusing completed DeepL translation: {audio_path}")
        continue
    source_code = deepl_source_code(result.get("language"))
    if source_code == target_code.split("-")[0]:
        print(f"Skipping {audio_path}: already in target language")
        continue
    if task == "transcribe" and source_code not in source_codes:
        print(f"Skipping {audio_path}: DeepL does not support source {source_code or 'unknown'}")
        continue

    # Translate the same cleaned transcript Step 4 wrote, not the raw loops.
    segments = result.get("segments", [])
    source_units = build_translation_units(
        clean_segments(segments) if cleanup_repetitions else segments)
    saved_segments = existing.get("segments", [])
    if existing.get("translation_status") == "in_progress" and not valid_resume_prefix(
            saved_segments, source_units):
        print(f"⚠️ Discarding inconsistent DeepL checkpoint entry: {audio_path}")
        existing = {}
    translated = existing if existing.get("translation_status") == "in_progress" else {
        "text": "", "segments": [], "language": deepl_target_language,
        "translation_status": "in_progress"}
    translated_results[audio_path] = translated

    deepl_batch_size = 50
    completed_segments = min(len(translated.get("segments", [])), len(source_units))
    for start in range(completed_segments, len(source_units), deepl_batch_size):
        batch = source_units[start:start + deepl_batch_size]
        texts = [unit["text"] for unit in batch]
        request = dict(
            text=texts,
            source_lang=source_code if task == "transcribe" else None,
            target_lang=target_code, formality=formality,
            # Include the lines before the batch so its first lines keep their context.
            context="\n".join(unit["text"] for unit in source_units[max(0, start - 10):start + len(batch)])
            if share_context else None,
            split_sentences="nonewlines",
            # Next-gen model is noticeably better for JA; falls back if unavailable.
            model_type="prefer_quality_optimized",
        )
        # deepl.Translator already retries 429 and 5xx responses with backoff.
        try:
            responses = translator.translate_text(**request, custom_instructions=custom_instructions)
        except (deepl.QuotaExceededException, deepl.AuthorizationException):
            raise
        except deepl.DeepLException as error:
            if not custom_instructions:
                raise
            print(f"ℹ️ DeepL rejected the style hint ({error}); translating without it.")
            custom_instructions = None
            responses = translator.translate_text(**request)
        if not isinstance(responses, list):
            responses = [responses]
        if len(responses) != len(batch):
            raise RuntimeError(f"DeepL returned {len(responses)} results for {len(batch)} segments")
        for unit, response in zip(batch, responses):
            translated["segments"].append({
                "id": unit.get("id"), "start": unit["start"], "end": unit["end"],
                "text": response.text.strip(),
            })
        translated["text"] = "\n".join(item["text"] for item in translated["segments"])
        atomic_deepl_checkpoint(translated_results)
        print(f"Translated {min(start + deepl_batch_size, len(source_units))}/{len(source_units)} segments")
    translated["translation_status"] = "complete"
    atomic_deepl_checkpoint(translated_results)

name_map = output_name_map(list(translated_results))
for audio_path, translated in translated_results.items():
    if translated.get("translation_status") != "complete":
        continue
    base = f"{name_map[audio_path]}-{target_code.lower()}"
    base = reserve_output_name(base, output_formats, output_dir,
                               overwrite=overwrite_existing)
    for output_format in output_formats:
        # raw: the source was cleaned before translation; cleaning the translation again
        # would drop a line that happens to translate like the one before it.
        path = write_result({**translated, "segments": subtitle_segments(translated["segments"])},
                            output_format, base, raw=True)
        if Path(path).suffix.lower() in {".srt", ".vtt"}:
            normalize_subtitle(path)
        print("✅", path)
print(f"\n✅ DeepL outputs saved in {output_dir}")
